In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

import sys 
import torch
from pathlib import Path

from satchange import (
    BCEDiceLoss,
    BinaryChangeMetrics,
    fit_binary_change_model,
)

from satchange.models import FCSiamDiff

from satchange.optimization import (
    run_parameter_search,
)

from torch.utils.data import DataLoader
from satchange.utils import load_config
from satchange.data import (
    RandomFlipRotate,
    create_dataset_dataloader,
    prepare_levir_cd,
)
from satchange.experiments import (
    load_fc_siam_diff_experiment
)

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Selected GPU:", torch.cuda.get_device_name(0))

Python executable: .venv/bin/python
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyTorch: 2.12.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
Visible GPU count: 4
Selected GPU: NVIDIA A100-SXM4-80GB


In [4]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

full_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_full.yaml"
)

resized_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_resized.yaml"
)


print("The project root: ", project_root)

print("Full config:", full_config_path)
print("Full config exists:", full_config_path.exists())

print("Resized config:", resized_config_path)
print("Resized config exists:", resized_config_path.exists())

The project root:  .
Full config: configs/fc_siam_diff_full.yaml
Full config exists: True
Resized config: configs/fc_siam_diff_resized.yaml
Resized config exists: True


### Resized Dataset

In [5]:
EXPERIMENT = "resized"
# Available "full" and "resized"


config, config_path = load_fc_siam_diff_experiment(
    experiment=EXPERIMENT,
    project_root=project_root,
)

print("Experiment:", EXPERIMENT)
print("Configuration:", config_path)
print("Image size:", config["data"]["image_size"])
print("Batch size:", config["data"]["batch_size"])
print("Output directory:", config["output"]["directory"])

Experiment: resized
Configuration: configs/fc_siam_diff_resized.yaml
Image size: 256
Batch size: 16
Output directory: artifacts/fc_siam_diff_resized


In [6]:
# path to the dataset
dataset_root = prepare_levir_cd(
    config=config,
    project_root=project_root,
)

print("Ready dataset root:", dataset_root)

Download directory is not empty: data/downloads/levir_cd
Skipping download.
Split is already extracted: data/LEVIR-CD/train
Skipping extraction.
Split is already extracted: data/LEVIR-CD/val
Skipping extraction.
Split is already extracted: data/LEVIR-CD/test
Skipping extraction.

Split:         train
Before images: 445
After images:  445
Labels:        445

Split:         val
Before images: 64
After images:  64
Labels:        64

Split:         test
Before images: 128
After images:  128
Labels:        128

Total samples: 637
LEVIR-CD structure is valid.
Ready dataset root: data/LEVIR-CD


In [7]:
train_transform = RandomFlipRotate(
    horizontal_flip_probability = 0.5,
    vertical_flip_probability = 0.5,
)

train_dataset, train_loader = create_dataset_dataloader(
    dataset_root=dataset_root,
    split="train",
    image_size=config["data"]["image_size"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    transform=train_transform,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
    seed=config["project"]["seed"],
)

validation_dataset, validation_loader = create_dataset_dataloader(
    dataset_root=dataset_root,
    split="val",
    image_size=config["data"]["image_size"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

test_dataset, test_loader = create_dataset_dataloader(
    dataset_root=dataset_root,
    split="test",
    image_size=config["data"]["image_size"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

In [8]:
RANDOM_SEED = config["project"]["seed"]

torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


model = FCSiamDiff(
    in_channels=(
        config["model"]["parameters"]["in_channels"]
    ),
    out_channels=(
        config["model"]["parameters"]["out_channels"]
    ),
    base_channels=(
        config["model"]["parameters"]["base_channels"]
    ),
).to(device)


loss_function = BCEDiceLoss(
    bce_weight=(
        config["training"]["loss"]["bce_weight"]
    ),
    dice_weight=(
        config["training"]["loss"]["dice_weight"]
    ),
    pos_weight=5,
).to(device)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=(
        config["training"]["weight_decay"]
    ),
)

In [9]:
THRESHOLDS = [
    round(value / 100, 2)
    for value in range(5, 96, 5)
]

batch_size = config["data"]["batch_size"]
learning_rate = config["training"]["learning_rate"]
seed = RANDOM_SEED
positive_weight = 10

MODEL_NAME = "fc_siam_diff"

RUN_NAME = (
    f"levir_{EXPERIMENT}_"
    f"pw{positive_weight:g}_"
    f"lr{learning_rate:g}_"
    f"bs{batch_size}_"
    f"seed{seed}"
)

OUTPUT_ROOT = (
    project_root
    / "artifacts"
    / "training_runs"
)

RUN_DIRECTORY = (
    OUTPUT_ROOT
    / MODEL_NAME
    / RUN_NAME
)

RESUME_CHECKPOINT = (
    RUN_DIRECTORY
    / "last.pt"
)

if not RESUME_CHECKPOINT.is_file():
    raise FileNotFoundError(
        "Resume checkpoint was not found: "
        f"{RESUME_CHECKPOINT}"
    )

print("Run name:", RUN_NAME)
print("Resume checkpoint:", RESUME_CHECKPOINT)

checkpoint_information = torch.load(
    RESUME_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print(
    "Saved epoch:",
    checkpoint_information["epoch"],
)

print(
    "Saved model:",
    checkpoint_information.get(
        "model_name"
    ),
)

print(
    "Saved run:",
    checkpoint_information.get(
        "run_name"
    ),
)

print(
    "Best metrics:",
    checkpoint_information.get(
        "best_metrics"
    ),
)

print(
    "Contains optimizer state:",
    "optimizer_state_dict"
    in checkpoint_information,
)



Run name: levir_resized_pw10_lr0.001_bs16_seed42
Resume checkpoint: artifacts/training_runs/fc_siam_diff/levir_resized_pw10_lr0.001_bs16_seed42/last.pt
Saved epoch: 110
Saved model: fc_siam_diff
Saved run: levir_resized_pw10_lr0.001_bs16_seed42
Best metrics: {'epoch': 103, 'train_loss': 0.36003440692183675, 'validation_loss': 0.36435065418481827, 'best_threshold': 0.85, 'validation_precision': 0.7088226011818607, 'validation_recall': 0.74000056806885, 'validation_f1': 0.7240761173164949, 'validation_iou': 0.5674916248088627, 'validation_accuracy': 0.9763295650482178, 'true_positive': 130266, 'false_positive': 53512, 'false_negative': 45769, 'true_negative': 3964757, 'f1_at_05': 0.6830171111425533, 'mean_changed_probability': 0.8214275348297214, 'mean_unchanged_probability': 0.03312555253502814, 'probability_gap': 0.7883019822946933}
Contains optimizer state: True


In [10]:
SEARCH_SPACE = {
    "learning_rate": [
        3e-4,
        1e-3,
        3e-3,
    ],
    "scheduler": [
        "none",
        "reduce_on_plateau",
    ],
}


FIXED_PARAMETERS = {
    "positive_weight": 10,
    "bce_weight": (
        config["training"]["loss"][
            "bce_weight"
        ]
    ),
    "dice_weight": (
        config["training"]["loss"][
            "dice_weight"
        ]
    ),
    "weight_decay": (
        config["training"][
            "weight_decay"
        ]
    ),
    "batch_size": (
        config["data"][
            "batch_size"
        ]
    ),
    "image_size": (
        config["data"][
            "image_size"
        ]
    ),
    "num_workers": (
        config["data"][
            "num_workers"
        ]
    ),
    "seed": (
        config["project"]["seed"]
    ),
    "model_parameters": dict(
        config["model"]["parameters"]
    ),
}

In [11]:
results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name="fc_siam_diff",
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=60,
    output_root=(
        project_root
        / "artifacts"
        / "hyperparameter_search"
    ),
    search_name=(
        "stage1_lr_scheduler_smoke"
    ),
    gpu_ids=[
        0,
        1,
        2,
        3,
    ],
    checkpoint_interval=5,
)

Trials: 6
GPUs: [0, 1, 2, 3]



[Trial 002 | GPU 2] Epoch 1/60
Train loss:       0.917677
Validation loss:  0.891368
Best threshold:   0.35
Precision:        0.045956
Recall:           0.977164
F1:               0.087784
IoU:              0.045907

[Trial 003 | GPU 3] Epoch 1/60
Train loss:       0.917677
Validation loss:  0.891368
Best threshold:   0.35
Precision:        0.045956
Recall:           0.977164
F1:               0.087784
IoU:              0.045907

[Trial 001 | GPU 1] Epoch 1/60
Train loss:       0.923454
Validation loss:  0.904910
Best threshold:   0.05
Precision:        0.041970
Recall:           1.000000
F1:               0.080559
IoU:              0.041970

[Trial 000 | GPU 0] Epoch 1/60
Train loss:       0.923454
Validation loss:  0.904910
Best threshold:   0.05
Precision:        0.041970
Recall:           1.000000
F1:               0.080559
IoU:              0.041970

[Trial 003 | GPU 3] Epoch 2/60
Train loss:       0.905838
Validation loss:  0.872085
Best threshold:   0.35
Precision:        0.122

In [12]:
# Sort trials from best to worst by validation F1.
sorted_results = sorted(
    results,
    key=lambda result: result["best_f1"],
    reverse=True,
)


print()
print("=" * 120)
print("STAGE 1: LEARNING RATE × SCHEDULER RESULTS")
print("=" * 120)

print(
    f"{'Trial':>5} | "
    f"{'LR':>10} | "
    f"{'Scheduler':>18} | "
    f"{'Epoch':>5} | "
    f"{'Threshold':>9} | "
    f"{'Precision':>9} | "
    f"{'Recall':>9} | "
    f"{'F1':>9} | "
    f"{'IoU':>9} | "
    f"{'GPU h':>8} | "
    f"{'VRAM MB':>10}"
)

print("-" * 120)

for result in sorted_results:
    parameters = result["parameters"]

    print(
        f"{result['trial']:5d} | "
        f"{parameters['learning_rate']:10.1e} | "
        f"{parameters['scheduler']:>18} | "
        f"{result['best_epoch']:5d} | "
        f"{result['best_threshold']:9.2f} | "
        f"{result['best_precision']:9.4f} | "
        f"{result['best_recall']:9.4f} | "
        f"{result['best_f1']:9.4f} | "
        f"{result['best_iou']:9.4f} | "
        f"{result['gpu_hours']:8.3f} | "
        f"{result['peak_gpu_memory_mb']:10.1f}"
    )


STAGE 1: LEARNING RATE × SCHEDULER RESULTS
Trial |         LR |          Scheduler | Epoch | Threshold | Precision |    Recall |        F1 |       IoU |    GPU h |    VRAM MB
------------------------------------------------------------------------------------------------------------------------
    3 |    1.0e-03 |  reduce_on_plateau |    57 |      0.85 |    0.6185 |    0.7110 |    0.6616 |    0.4943 |    1.042 |     1595.7
    2 |    1.0e-03 |               none |    57 |      0.85 |    0.6185 |    0.7110 |    0.6616 |    0.4943 |    1.055 |     1595.7
    4 |    3.0e-03 |               none |    60 |      0.90 |    0.6347 |    0.6167 |    0.6256 |    0.4552 |    0.746 |     1595.7
    5 |    3.0e-03 |  reduce_on_plateau |    60 |      0.90 |    0.6347 |    0.6167 |    0.6256 |    0.4552 |    0.746 |     1595.7
    1 |    3.0e-04 |  reduce_on_plateau |    58 |      0.85 |    0.5160 |    0.6060 |    0.5574 |    0.3864 |    1.043 |     1595.7
    0 |    3.0e-04 |               none |  

In [13]:
best_result = sorted_results[0]
best_parameters = best_result["parameters"]

print()
print("=" * 80)
print("BEST TRIAL")
print("=" * 80)

print(
    "Trial:",
    best_result["trial"],
)

print(
    "Learning rate:",
    best_parameters["learning_rate"],
)

print(
    "Scheduler:",
    best_parameters["scheduler"],
)

print(
    "Best epoch:",
    best_result["best_epoch"],
)

print(
    "Best threshold:",
    best_result["best_threshold"],
)

print(
    "Precision:",
    f"{best_result['best_precision']:.6f}",
)

print(
    "Recall:",
    f"{best_result['best_recall']:.6f}",
)

print(
    "F1:",
    f"{best_result['best_f1']:.6f}",
)

print(
    "IoU:",
    f"{best_result['best_iou']:.6f}",
)

print(
    "Validation loss:",
    f"{best_result['best_validation_loss']:.6f}",
)

print(
    "Training time:",
    f"{best_result['training_seconds'] / 60:.2f} min",
)

print(
    "GPU hours:",
    f"{best_result['gpu_hours']:.3f}",
)

print(
    "Peak GPU memory:",
    f"{best_result['peak_gpu_memory_mb']:.1f} MB",
)


BEST TRIAL
Trial: 3
Learning rate: 0.001
Scheduler: reduce_on_plateau
Best epoch: 57
Best threshold: 0.85
Precision: 0.618527
Recall: 0.711035
F1: 0.661563
IoU: 0.494280
Validation loss: 0.451196
Training time: 62.49 min
GPU hours: 1.042
Peak GPU memory: 1595.7 MB


In [14]:
learning_rates = sorted(
    {
        result["parameters"]["learning_rate"]
        for result in results
    }
)

print()
print("=" * 100)
print("SCHEDULER EFFECT AT THE SAME LEARNING RATE")
print("=" * 100)

for learning_rate in learning_rates:
    same_lr_results = [
        result
        for result in results
        if result["parameters"]["learning_rate"]
        == learning_rate
    ]

    without_scheduler = next(
        (
            result
            for result in same_lr_results
            if result["parameters"]["scheduler"]
            == "none"
        ),
        None,
    )

    with_scheduler = next(
        (
            result
            for result in same_lr_results
            if result["parameters"]["scheduler"]
            == "reduce_on_plateau"
        ),
        None,
    )

    if (
        without_scheduler is None
        or with_scheduler is None
    ):
        continue

    f1_without = without_scheduler["best_f1"]
    f1_with = with_scheduler["best_f1"]

    f1_difference = (
        f1_with
        - f1_without
    )

    iou_difference = (
        with_scheduler["best_iou"]
        - without_scheduler["best_iou"]
    )

    print()
    print(
        f"Learning rate: {learning_rate:g}"
    )

    print(
        "Without scheduler:"
    )

    print(
        f"  Best F1:   {f1_without:.6f}"
    )

    print(
        f"  Best IoU:  "
        f"{without_scheduler['best_iou']:.6f}"
    )

    print(
        f"  Best epoch: "
        f"{without_scheduler['best_epoch']}"
    )

    print(
        "With ReduceLROnPlateau:"
    )

    print(
        f"  Best F1:   {f1_with:.6f}"
    )

    print(
        f"  Best IoU:  "
        f"{with_scheduler['best_iou']:.6f}"
    )

    print(
        f"  Best epoch: "
        f"{with_scheduler['best_epoch']}"
    )

    print(
        f"F1 difference:  {f1_difference:+.6f}"
    )

    print(
        f"IoU difference: {iou_difference:+.6f}"
    )

    if f1_difference > 0:
        print(
            "Result: scheduler improved F1."
        )

    elif f1_difference < 0:
        print(
            "Result: scheduler reduced F1."
        )

    else:
        print(
            "Result: scheduler produced the same best F1."
        )


SCHEDULER EFFECT AT THE SAME LEARNING RATE

Learning rate: 0.0003
Without scheduler:
  Best F1:   0.557383
  Best IoU:  0.386369
  Best epoch: 58
With ReduceLROnPlateau:
  Best F1:   0.557383
  Best IoU:  0.386369
  Best epoch: 58
F1 difference:  +0.000000
IoU difference: +0.000000
Result: scheduler produced the same best F1.

Learning rate: 0.001
Without scheduler:
  Best F1:   0.661563
  Best IoU:  0.494280
  Best epoch: 57
With ReduceLROnPlateau:
  Best F1:   0.661563
  Best IoU:  0.494280
  Best epoch: 57
F1 difference:  +0.000000
IoU difference: +0.000000
Result: scheduler produced the same best F1.

Learning rate: 0.003
Without scheduler:
  Best F1:   0.625594
  Best IoU:  0.455174
  Best epoch: 60
With ReduceLROnPlateau:
  Best F1:   0.625594
  Best IoU:  0.455174
  Best epoch: 60
F1 difference:  +0.000000
IoU difference: +0.000000
Result: scheduler produced the same best F1.


In [15]:
print()
print("=" * 100)
print("BEST RESULT FOR EACH INITIAL LEARNING RATE")
print("=" * 100)

best_by_learning_rate = []

for learning_rate in learning_rates:
    candidates = [
        result
        for result in results
        if result["parameters"]["learning_rate"]
        == learning_rate
    ]

    best_for_lr = max(
        candidates,
        key=lambda result: result["best_f1"],
    )

    best_by_learning_rate.append(
        best_for_lr
    )

    print(
        f"LR {learning_rate:.1e} | "
        f"F1 {best_for_lr['best_f1']:.6f} | "
        f"IoU {best_for_lr['best_iou']:.6f} | "
        f"Scheduler "
        f"{best_for_lr['parameters']['scheduler']} | "
        f"Epoch {best_for_lr['best_epoch']}"
    )


best_learning_rate_result = max(
    best_by_learning_rate,
    key=lambda result: result["best_f1"],
)

print()
print(
    "Best initial learning rate:",
    best_learning_rate_result[
        "parameters"
    ]["learning_rate"],
)

print(
    "Best scheduler:",
    best_learning_rate_result[
        "parameters"
    ]["scheduler"],
)

print(
    "Best F1:",
    f"{best_learning_rate_result['best_f1']:.6f}",
)


BEST RESULT FOR EACH INITIAL LEARNING RATE
LR 3.0e-04 | F1 0.557383 | IoU 0.386369 | Scheduler reduce_on_plateau | Epoch 58
LR 1.0e-03 | F1 0.661563 | IoU 0.494280 | Scheduler reduce_on_plateau | Epoch 57
LR 3.0e-03 | F1 0.625594 | IoU 0.455174 | Scheduler none | Epoch 60

Best initial learning rate: 0.001
Best scheduler: reduce_on_plateau
Best F1: 0.661563


In [16]:
total_gpu_hours = sum(
    result["gpu_hours"]
    for result in results
)

total_training_seconds = sum(
    result["training_seconds"]
    for result in results
)

average_seconds_per_epoch = (
    sum(
        result["seconds_per_epoch"]
        for result in results
    )
    / len(results)
)

maximum_peak_gpu_memory_mb = max(
    result["peak_gpu_memory_mb"]
    for result in results
)

print()
print("=" * 80)
print("COMPUTE COST")
print("=" * 80)

print(
    "Trials:",
    len(results),
)

print(
    "Total GPU hours:",
    f"{total_gpu_hours:.3f}",
)

print(
    "Sum of trial training time:",
    f"{total_training_seconds / 3600:.3f} h",
)

print(
    "Average seconds per epoch:",
    f"{average_seconds_per_epoch:.2f}",
)

print(
    "Maximum peak GPU memory:",
    f"{maximum_peak_gpu_memory_mb:.1f} MB",
)


COMPUTE COST
Trials: 6
Total GPU hours: 5.676
Sum of trial training time: 5.676 h
Average seconds per epoch: 56.76
Maximum peak GPU memory: 1595.7 MB


In [17]:
STAGE_1_BEST = {
    "learning_rate": (
        best_result["parameters"][
            "learning_rate"
        ]
    ),
    "scheduler": (
        best_result["parameters"][
            "scheduler"
        ]
    ),
}

print()
print(
    "Parameters selected for Stage 2:",
    STAGE_1_BEST,
)


Parameters selected for Stage 2: {'learning_rate': 0.001, 'scheduler': 'reduce_on_plateau'}


### Full sized Dataset

In [18]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [19]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")

In [20]:
# before_features = model._encode(before)
# after_features = model._encode(after)

# feature_differences = model._compute_feature_differences(
#     before_features,
#     after_features,
# )

# decoded_features = model._decode(
#     feature_differences
# )

# print(decoded_features.shape)

In [21]:
# # switches the model into training mode
# model.train()
# # and
# model.eval()

# for batch in train_loader:
#     # cleares gradients left from previous batches 'cause PyTorch accumulates gradients
#     optimizer.zero_grad()

#     # the prediction and computation graph
#     prediction = model(
#         batch["before"],
#         batch["after"],
#     )

#     # computing the loss (how different the prediction from the real mask)
#     loss = loss_function(
#         prediction,
#         batch["mask"],
#     )

#     # computing the gradient (what change to reduce the loss)
#     loss.backward()

#     # update the weights using gradients
#     optimizer.step()


# # PyTorch cycle that does not effect the model. It's used to evaluate but not to train.
# with torch.no_grad():
#     for batch in validation_loader:
#         prediction = model(
#             batch["before"],
#             batch["after"],
#         )

#         # computing the loss
#         loss = loss_function(
#             prediction,
#             batch["mask"]
#         )

### Full sized Dataset

In [22]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [23]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")